# Notebook 9bis — BERTopic: NLP-Based Clustering on Clinical Text

## Purpose

This notebook tests BERTopic as an alternative to the tabular HDBSCAN pipeline. Clinical notes are converted to natural-language sentences, embedded with a medical sentence transformer, and clustered via UMAP + HDBSCAN to assess whether text-based representations of resource utilisation yield coherent patient groups. This approach was ultimately not retained as the primary method.

---

## Pipeline Overview

### 1. Embedding model
A medical sentence transformer ("embeddinggemma-300m-medical") and/or a longformer ("yikuan8/Clinical-Longformer") are loaded and used to encode patient-level clinical text. The batch-encoding pipeline applies a maximum token length of 512 and processes patients in batches of 256.

### 2. Text construction — `build_clinical_text()`
Each patient row is converted to a structured English sentence describing their ED resource utilisation. Two configurations are supported:
- **CONFIG_SIMPLE** (boolean imaging): "The patient underwent an X-ray and a CT scan."
- **CONFIG_DETAILED** (categorical imaging): lists specific exam labels by modality.

Sections cover imaging, biology (blood tests, cultures, blood gas, lumbar puncture), exam count summaries, procedures (EKG), and discharge disposition.

### 3. UMAP dimensionality reduction
Embeddings are reduced to 10 dimensions using UMAP (`n_neighbors=15`, `min_dist=0.0`), following the same configuration as the primary tabular pipeline.

### 4. HDBSCAN grid sweep
A grid search over `min_cluster_size` (2–6% of N) and `min_samples` (1–20% of mcs) is run with the EOM cluster selection method. Each run is evaluated on a combined quality score weighting silhouette (0.35), cluster stability (0.25), outlier rate (−0.25), and number-of-clusters proximity to the target range (0.15).

### 5. BERTopic integration
BERTopic wraps the UMAP + HDBSCAN pipeline and uses c-TF-IDF with a `CountVectorizer` (custom clinical stop words removed) to generate topic representations for each cluster. Word clouds and topic distribution plots are produced for visual inspection.

> **Note:** BERTopic yielded lower silhouette scores and less clinically coherent clusters than the tabular Scenario 2 approach and was not used for primary results.

In [2]:
# ============================================================
# 1. IMPORTS, PATHS & HYPERPARAMETERS
# ============================================================

import os
import warnings
import logging
import pandas as pd
import numpy as np
import torch
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from wordcloud import WordCloud
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# ── LOGGER ────────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
log = logging.getLogger(__name__)

# ── THREADING & DEVICE ────────────────────────────────────────────────────────
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "128"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")

# ── PATHS ─────────────────────────────────────────────────────────────────────
CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/BERTopic/sentence_transformer"
#OUTPUT_DIR = "Results/BERTopic/longformer"
# ── EMBEDDING MODEL ───────────────────────────────────────────────────────────
MODEL_NAME  = "sentence-transformers/embeddinggemma-300m-medical"
#MODEL_NAME  = "yikuan8/Clinical-Longformer"
embed_model = SentenceTransformer(MODEL_NAME)
embed_model.encode(["test"], show_progress_bar=False)   # warm-up
log.info(f"Model ready: {MODEL_NAME}")

# ── HYPERPARAMETERS UMAP───────────────────────────────────────────────────────────
BATCH_SIZE     = 256
MAX_LENGTH     = 512

UMAP_DIM       = 10
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST  = 0.0

# ── HYPERPARAMETERS HDBSCAN ───────────────
MCS_PCT_VALUES    = [0.02, 0.03, 0.04, 0.05, 0.06]
MS_PCT_MCS_VALUES = [0.01, 0.05, 0.10, 0.20]
MS_FIXED_VALUES   = [1, 5, 10, 15]

N_MIN_CLUSTERS = 5
N_MAX_CLUSTERS = 15

W_SILHOUETTE = 0.35
W_STABILITY  = 0.25
W_OUTLIER    = 0.25
W_NCLUSTERS  = 0.15

HDBSCAN_CLUSTER_METHOD = "eom"

# ============================================================
# 2. COLUMN DEFINITIONS & CONFIGS
# ============================================================

# ── IMAGING ───────────────────────────────────────────────────────────────────

# Boolean imaging flags — used in CONFIG_SIMPLE
IMAGING_COLS_BOOL = {
    'has_ultrasound': "an ultrasound",
    'has_ct_scan':    "a CT scan",
    'has_xray':       "an X-ray",
    'has_mri':        "an MRI",
}

# Categorical imaging columns by modality — used in CONFIG_DETAILED
IMAGING_COLS_DETAILED = {
    'ultrasound': ['ultrasound_1', 'ultrasound_2'],
    'ct_scan':    ['ct_scan_1', 'ct_scan_2', 'ct_scan_3'],
    'xray':       ['xray_1', 'xray_2', 'xray_3'],
    'mri':        ['mri_1', 'mri_2'],
}

# ── BIOLOGY ───────────────────────────────────────────────────────────────────

BIO_COLS = {
    # Venous / blood tests
    'is_hemoglobine':     "hemoglobin",
    'is_leucocytes':      "white blood cell count",
    'is_formule_leuco':   "differential leukocyte count",
    'is_urea':            "blood urea",
    'is_creatinine':      "creatinine",
    'is_sodium':          "sodium",
    'is_potassium':       "potassium",
    'is_platelets':       "platelet count",
    'is_pt':              "prothrombin time",
    'is_aptt':            "aPTT",
    'is_calcium':         "calcium",
    'is_ck':              "CK",
    'is_lactates':        "lactate",
    'is_troponine':       "troponin",
    'is_bnp':             "BNP",
    'is_ckmb':            "CK-MB",
    'is_ddimer':          "D-dimer",
    'is_crp':             "CRP",
    'is_pct':             "procalcitonin",
    'is_alat':            "ALT",
    'is_asat':            "AST",
    'is_bili_total':      "total bilirubin",
    'is_lipase':          "lipase",
    'is_alp':             "ALP",
    'is_iron':            "serum iron",
    'is_ferritin':        "ferritin",
    'is_calcium_ionized': "ionized calcium",
    'is_aXa_aIIa':        "anti-Xa/anti-IIa activity",
    'is_fibrinogen':      "fibrinogen",
    # Special exams
    'has_blood_test':     "blood work",
    'has_culture':        "microbiological cultures",
    'has_lumbar_puncture':"lumbar puncture",
    'has_blood_gas':      "arterial blood gas",
}

# ── PROCEDURES ────────────────────────────────────────────────────────────────

PROCEDURE_COLS = {
    'had_ekg': "an EKG",
}

# ── COLUMN GROUPS ─────────────────────────────────────────────────────────────

cols_venous = [c for c in BIO_COLS if c.startswith('is_')]

cols_special_bio = [
    'has_blood_gas', 'has_culture', 'has_lumbar_puncture',
]

cols_imaging_cat = [
    col
    for cols in IMAGING_COLS_DETAILED.values()
    for col in cols
] + ['radio_interventional_1', 'nuclear_medicine_1']

cols_bio_bin   = cols_venous + cols_special_bio   # all binary biology flags
cols_quanti    = ['imaging_exam_count', 'bio_exam_count']
cols_multi_cat = cols_imaging_cat                 # alias for Top3 signature

# ── SCENARIOS ─────────────────────────────────────────────────────────────────

_disposition  = ['hospitalization', 'observation_unit']
_imaging_bool = list(IMAGING_COLS_BOOL)
_special_bio  = list(k for k in BIO_COLS if k.startswith('has_'))
_counts       = cols_quanti

SCENARIOS = {
    # Scenario 2: aggregated boolean imaging + biology flags + disposition
    "scenario_2": (
        _imaging_bool + _special_bio
        + list(PROCEDURE_COLS)
        + _disposition + _counts
    ),
    # Scenario 3: detailed venous + categorical imaging + disposition
    "scenario_3": (
        cols_venous + cols_imaging_cat + _imaging_bool + _special_bio
        + list(PROCEDURE_COLS)
        + _disposition + _counts
    ),
}

# ── TEXT-BUILDING CONFIGS ─────────────────────────────────────────────────────

CONFIG_SIMPLE = {
    "imaging_mode":   "boolean",
    "imaging_cols":   IMAGING_COLS_BOOL,
    "bio_cols":       BIO_COLS,
    "procedure_cols": PROCEDURE_COLS,
    "has_disposition": True,
}

CONFIG_DETAILED = {
    "imaging_mode":   "categorical",
    "imaging_cols":   IMAGING_COLS_DETAILED,
    "bio_cols":       BIO_COLS,
    "procedure_cols": PROCEDURE_COLS,
    "has_disposition": True,
}

SCENARIO_CONFIGS = {
    "scenario_2": {"config": CONFIG_SIMPLE,   "cols": SCENARIOS["scenario_2"]},
    "scenario_3": {"config": CONFIG_DETAILED, "cols": SCENARIOS["scenario_3"]},
}

# ── STOP WORDS ────────────────────────────────────────────────────────────────

CLINICAL_STOP_WORDS = list(ENGLISH_STOP_WORDS) + [
    # Verbs and sentence structure
    "patient", "underwent", "performed", "measured",
    "had", "was", "were", "placed", "subsequently",
]

# %%


2026-05-20 10:40:27,209 — INFO — Device: cuda
2026-05-20 10:40:27,211 — INFO — Use pytorch device_name: cuda:0
2026-05-20 10:40:27,212 — INFO — Load pretrained SentenceTransformer: sentence-transformers/embeddinggemma-300m-medical
2026-05-20 10:40:27,508 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 10:40:27,540 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/modules.json "HTTP/1.1 200 OK"
2026-05-20 10:40:27,675 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 10:40:27,676 — WARNING — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and 

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

2026-05-20 10:40:29,372 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 10:40:29,404 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/config.json "HTTP/1.1 200 OK"
2026-05-20 10:40:29,546 — INFO — HTTP Request: HEAD https://huggingface.co/sentence-transformers/embeddinggemma-300m-medical/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 10:40:29,577 — INFO — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/embeddinggemma-300m-medical/ffb7eedc5047b2e26bcbccbf09c435ae4a80a08d/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-20 10:40:29,742 — INFO — HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/embeddinggemma-300m-medical/tree/main/additional_chat_templates

In [3]:

# ============================================================
# 3. TEXT BUILDER HELPERS & FUNCTIONS
# ============================================================

def _is_present(val) -> bool:
    """Returns True if val represents a positive / present finding."""
    if val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except Exception:
        pass
    if str(val).upper() in ["NONE", "NAN", "", "0", "FALSE"]:
        return False
    return val in (1, 1.0, True) or isinstance(val, str)


def _clean_label(val):
    """Normalises a raw categorical value to a readable string, or None."""
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except Exception:
        pass
    val = str(val).strip()
    if val.upper() in ["NONE", "NAN", "", "0"]:
        return None
    return val.lower().replace("_", " ")


def _join_list(items: list) -> str:
    """['a', 'b', 'c']  ->  'a, b and c'"""
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + f" and {items[-1]}"


def _with_article(s: str) -> str:
    """Prepends 'an' or 'a' based on first letter."""
    return f"an {s}" if s[0] in "aeiouAEIOU" else f"a {s}"


def _imaging_sentence_boolean(row, imaging_cols: dict, active_cols: list = None) -> str | None:
    done = [
        label for col, label in imaging_cols.items()
        if _is_present(row.get(col))
        and (active_cols is None or col in active_cols)  # <-- filtre
    ]
    if not done:
        return None
    return f"The patient underwent {_join_list(done)}."


def _imaging_sentence_categorical(row, imaging_cols: dict, active_cols: list = None) -> str | None:
    exams = []
    for modality, cols in imaging_cols.items():
        for col in cols:
            if active_cols is not None and col not in active_cols:  # <-- filtre
                continue
            val = _clean_label(row.get(col))
            if val is not None:
                exams.append(_with_article(val))
    if not exams:
        return None
    return f"The patient underwent {_join_list(exams)}."


def build_clinical_text(row, config: dict, active_cols: list = None) -> str:
    sentences = []

    # ── 1. IMAGING ───────────────────────────────────────────────
    if config["imaging_mode"] == "categorical":
        imaging_sentence = _imaging_sentence_categorical(row, config["imaging_cols"], active_cols)
    else:
        imaging_sentence = _imaging_sentence_boolean(row, config["imaging_cols"], active_cols)

    imaging_count = int(row.get('imaging_exam_count') or 0)
    if imaging_sentence:
        sentences.append(imaging_sentence)
    elif imaging_count > 0:
        sentences.append(f"The patient underwent {imaging_count} imaging exam(s).")

    # ── 2. BIOLOGICAL EXAMS ──────────────────────────────────────
    # Filtre sur active_cols si fourni
    bio_cols_active = {
        col: label for col, label in config["bio_cols"].items()
        if active_cols is None or col in active_cols
    }
    bio_done  = [label for col, label in bio_cols_active.items() if _is_present(row.get(col))]
    bio_count = int(row.get('bio_exam_count') or 0)

    if bio_done:
        verb = "were" if len(bio_done) > 1 else "was"
        sentences.append(f"{_join_list(bio_done).capitalize()} {verb} measured.")
    elif bio_count > 0:
        sentences.append(f"{bio_count} biological exam(s) were performed.")

    # ── 3. EXAM COUNT SUMMARY ────────────────────────────────────
    count_parts = []
    if imaging_count > 0:
        count_parts.append(f"{imaging_count} imaging exam(s)")
    if bio_count > 0:
        count_parts.append(f"{bio_count} biological exam(s)")
    if count_parts:
        sentences.append(f"In total, the patient had {_join_list(count_parts)}.")

    # ── 4. PROCEDURES ────────────────────────────────────────────
    proc_cols_active = {
        col: label for col, label in config["procedure_cols"].items()
        if active_cols is None or col in active_cols
    }
    procedures = [label for col, label in proc_cols_active.items() if _is_present(row.get(col))]
    if procedures:
        sentences.append(f"The patient had {_join_list(procedures)} performed.")


    # ── 5. DISPOSITION ───────────────────────────────────────────
    if config.get("has_disposition"):
        observation  = _is_present(row.get('observation_unit'))
        hospitalized = _is_present(row.get('hospitalization'))

        if observation and hospitalized:
            sentences.append("The patient was placed in the observation unit and then admitted to the hospital.")
        elif observation:
            sentences.append("The patient was placed in the observation unit and subsequently discharged.")
        elif hospitalized:
            sentences.append("The patient was admitted to the hospital.")
        elif row.get('hospitalization') == 0:
            sentences.append("The patient was discharged.")

    return " ".join(sentences) if sentences else "No significant resource utilization was documented."

In [4]:

# ============================================================
# 4. CLUSTERING METRICS FUNCTIONS
# ============================================================

def calculate_metrics(umap_embeddings, labels, hdbscan_model):
    """
    Computes clustering quality metrics:
      - Silhouette score  : separation between clusters (higher = better)
      - Stability         : mean HDBSCAN cluster persistence (higher = denser)
      - Outlier rate      : fraction of noise points (-1)
      - Combined score    : weighted combination of the three metrics

    Returns (silhouette, stability, outlier_rate, combined_score).
    Returns zeros if clustering is degenerate (< 2 clusters).
    """
    mask       = labels != -1
    n_valid    = np.sum(mask)
    n_clusters = len(np.unique(labels[mask]))

    if n_valid < 2 or n_clusters < 2:
        log.warning("Degenerate clustering — returning zero scores")
        return 0.0, 0.0, 1.0, 0.0

    silhouette   = silhouette_score(umap_embeddings[mask], labels[mask], sample_size=3000)
    outlier_rate = np.sum(labels == -1) / len(labels)
    stability    = np.mean(hdbscan_model.cluster_persistence_)
    combined     = (silhouette * 0.4) + (stability * 0.4) + ((1 - outlier_rate) * 0.2)

    return silhouette, stability, outlier_rate, combined


def make_mcs_ms_grid(N):
    """Builds a 2D grid of (mcs, min_samples) derived as % of N."""
    grid = []
    for mcs_pct in MCS_PCT_VALUES:
        mcs = max(10, int(N * mcs_pct))
        ms_values = sorted(set(
            MS_FIXED_VALUES + [max(1, int(mcs * p)) for p in MS_PCT_MCS_VALUES]
        ))
        for ms in ms_values:
            grid.append({
                "mcs":        mcs,
                "ms":         ms,
                "mcs_pct_N":  round(mcs / N * 100, 2),
                "ms_pct_mcs": round(ms / mcs * 100, 2),
            })
    return grid


def cluster_score(n, n_min=N_MIN_CLUSTERS, n_max=N_MAX_CLUSTERS):
    """Penalizes solutions with too few or too many clusters."""
    if n_min <= n <= n_max:
        return 1.0
    elif n < n_min:
        return max(0.0, (n - 2) / (n_min - 2))
    else:
        return max(0.0, (50 - n) / (50 - n_max))


def _minmax(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx != mn else s * 0


def run_grid_sweep(umap_embeddings, run_label, out_dir, N):
    """
    2D grid sweep over (mcs, min_samples) derived as % of N.
    Outputs: sweep_report.csv + one curve plot per mcs value.
    """
    grid = make_mcs_ms_grid(N)
    log.info(f"[{run_label}] Grid sweep: {len(grid)} combinations for N={N:,}")
    log.info(f"  mcs range: {int(N*MCS_PCT_VALUES[0]):,} – {int(N*MCS_PCT_VALUES[-1]):,}")

    results = []
    for params in tqdm(grid, desc=f"Sweep — {run_label}"):
        mcs = params["mcs"]
        ms  = params["ms"]

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs, min_samples=ms,
            metric="euclidean", cluster_selection_method=HDBSCAN_CLUSTER_METHOD,
            gen_min_span_tree=False,
        ).fit(umap_embeddings)

        labels       = clusterer.labels_
        n_clusters   = len(np.unique(labels[labels >= 0]))
        outlier_rate = float(np.mean(labels == -1))
        sil          = (silhouette_score(
                            umap_embeddings[labels >= 0], labels[labels >= 0],
                            sample_size=min(3000, np.sum(labels >= 0)))
                        if n_clusters >= 2 else np.nan)
        stability    = (float(np.mean(clusterer.cluster_persistence_))
                        if len(clusterer.cluster_persistence_) > 0 else np.nan)

        results.append({
            "mcs":          mcs,
            "mcs_pct_N":    params["mcs_pct_N"],
            "ms":           ms,
            "ms_pct_mcs":   params["ms_pct_mcs"],
            "n_clusters":   n_clusters,
            "outlier_rate": outlier_rate,
            "outlier_pct":  round(outlier_rate * 100, 1),
            "silhouette":   round(sil, 4) if not np.isnan(sil) else np.nan,
            "stability":    round(stability, 4) if not np.isnan(stability) else np.nan,
        })

        sil_str = f"{sil:.3f}" if not np.isnan(sil) else "nan"
        log.info(f"  mcs={mcs:5d} ({params['mcs_pct_N']:.2f}%N) | "
                 f"ms={ms:3d} ({params['ms_pct_mcs']:.1f}%mcs) | "
                 f"clusters={n_clusters} | outliers={outlier_rate*100:.1f}% | sil={sil_str}")

    df_res = pd.DataFrame(results)

    # ── Combined score ────────────────────────────────────────
    df_res["combined_score"] = (
        W_SILHOUETTE * _minmax(df_res["silhouette"].fillna(0))
        + W_STABILITY  * _minmax(df_res["stability"].fillna(0))
        - W_OUTLIER    * df_res["outlier_rate"]
        + W_NCLUSTERS  * df_res["n_clusters"].apply(cluster_score)
    ).round(4)

    # ── Save CSV ──────────────────────────────────────────────
    os.makedirs(out_dir, exist_ok=True)
    df_res.to_csv(os.path.join(out_dir, f"sweep_report_{run_label}.csv"), index=False)

    # ── Curve plots (one per mcs value) ──────────────────────
    def _minmax_plot(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else pd.Series([0.5] * len(s), index=s.index)

    curve_colors = {
        "silhouette": "#2196F3",
        "stability":  "#4CAF50",
        "outlier":    "#F44336",
        "n_clusters": "#FF9800",
        "combined":   "#9C27B0",
    }

    for mcs_val in sorted(df_res["mcs"].unique()):
        df_mcs  = df_res[df_res["mcs"] == mcs_val].copy().sort_values("ms")
        mcs_pct = df_mcs["mcs_pct_N"].iloc[0]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(df_mcs["ms"], _minmax_plot(df_mcs["silhouette"].fillna(0)),
                color=curve_colors["silhouette"], marker="o", linewidth=2, label="Silhouette (norm)")
        ax.plot(df_mcs["ms"], _minmax_plot(df_mcs["stability"].fillna(0)),
                color=curve_colors["stability"],  marker="s", linewidth=2, label="Stability (norm)")
        ax.plot(df_mcs["ms"], 1 - df_mcs["outlier_rate"],
                color=curve_colors["outlier"],    marker="^", linewidth=2, label="1 - Outlier rate")
        ax.plot(df_mcs["ms"], df_mcs["n_clusters"].apply(cluster_score),
                color=curve_colors["n_clusters"], marker="D", linewidth=2, label="N clusters score")
        ax.plot(df_mcs["ms"], _minmax_plot(df_mcs["combined_score"]),
                color=curve_colors["combined"],   marker="*", linewidth=2.5,
                linestyle="--", markersize=10,    label="Combined score")

        ax.set_xlabel("min_samples (ms)", fontsize=11)
        ax.set_ylabel("Score (normalized to [0, 1])", fontsize=11)
        ax.set_title(
            f"Clustering scores vs. min_samples — {run_label}\n"
            f"mcs = {mcs_val} ({mcs_pct:.2f}% of N)",
            fontsize=12, fontweight="bold"
        )
        ax.legend(loc="upper right", fontsize=9)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=0.3)
        sns.despine(ax=ax)
        plt.tight_layout()
        plt.savefig(
            os.path.join(out_dir, f"sweep_curves_{run_label}_mcs{mcs_val}.png"),
            dpi=150
        )
        plt.close()

    best = df_res.loc[df_res["combined_score"].idxmax()]
    log.info(f"[{run_label}] Sweep complete — best: mcs={int(best['mcs'])} | "
             f"ms={int(best['ms'])} | clusters={int(best['n_clusters'])} | "
             f"combined={best['combined_score']:.4f}")

    return df_res

In [5]:

# ============================================================
#  5. VISUALIZATION FUNCTIONS
# ============================================================

def plot_wordclouds(df_valid, sc_name, out_dir):
    """Generates one wordcloud per cluster from clinical text."""
    clusters = sorted(df_valid["cluster"].unique())
    n        = len(clusters)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, cl in zip(axes, clusters):
        text = " ".join(df_valid[df_valid["cluster"] == cl]["clinical_text"].tolist())
        wc   = WordCloud(
            width=600, height=400,
            background_color="white",
            stopwords=set(CLINICAL_STOP_WORDS)
        ).generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Cluster {cl} (n={len(df_valid[df_valid['cluster']==cl])})")
    plt.suptitle(f"Wordclouds — {sc_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"wordclouds_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Wordclouds saved — {sc_name}")


def plot_heatmap(df_valid, c_bio, sc_name, out_dir):
    """Heatmap of binary bio feature frequency per cluster."""
    heat = df_valid.groupby("cluster")[c_bio].mean() * 100
    plt.figure(figsize=(max(12, len(c_bio) * 0.5), max(6, len(heat) * 0.6)))
    sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "%"})
    plt.title(f"Bio Feature Frequency per Cluster — {sc_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"heatmap_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Heatmap saved — {sc_name}")


def plot_radar(df_valid, c_bio, sc_name, out_dir):
    """Radar chart of binary bio profile per cluster."""
    means    = df_valid.groupby("cluster")[c_bio].mean() * 100
    cats     = c_bio
    n_cats   = len(cats)
    angles   = np.linspace(0, 2 * np.pi, n_cats, endpoint=False).tolist()
    angles  += angles[:1]

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    for cl, row in means.iterrows():
        values  = row.tolist() + row.tolist()[:1]
        ax.plot(angles, values, label=f"Cluster {cl}")
        ax.fill(angles, values, alpha=0.1)
    ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=7)
    ax.set_title(f"Bio Radar — {sc_name}", fontsize=13, fontweight="bold")
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"radar_{sc_name}.png"), dpi=120)
    plt.close()
    log.info(f"Radar chart saved — {sc_name}")


def plot_umap2d(umap_embeddings, best_labels, best_config, sc_name, out_dir):
    """
    Projects UMAP embeddings into 2D (global + local views) and plots clusters.
    Uses euclidean metric because embeddings are already UMAP-reduced.
    """
    log.info("UMAP 2D — global view (n_neighbors=30)...")
    umap_global = umap.UMAP(
        n_components=2, n_neighbors=30,
        min_dist=0.1, metric="euclidean", random_state=42
    ).fit_transform(umap_embeddings)

    log.info("UMAP 2D — local view (n_neighbors=10)...")
    umap_local = umap.UMAP(
        n_components=2, n_neighbors=10,
        min_dist=0.01, metric="euclidean", random_state=42
    ).fit_transform(umap_embeddings)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    title = (
        f"BEST MODEL : {sc_name.upper()}\n"
        f"Clusters: {int(best_config['n_clusters'])} | "
        f"MCS: {int(best_config['mcs'])} | "
        f"Silhouette: {best_config['silhouette']:.3f} | "
        f"Stability: {best_config['stability']:.3f} | "
        f"Outliers: {best_config['outlier_rate']*100:.1f}%"
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, coord, name in [
        (ax1, umap_global, "UMAP Global (n_neighbors=30)"),
        (ax2, umap_local,  "UMAP Local  (n_neighbors=10)"),
    ]:
        ax.scatter(
            coord[best_labels == -1, 0], coord[best_labels == -1, 1],
            s=1, color="lightgrey", alpha=0.2, label="Noise"
        )
        for cluster_id in sorted(set(best_labels) - {-1}):
            ax.scatter(
                coord[best_labels == cluster_id, 0],
                coord[best_labels == cluster_id, 1],
                s=4, alpha=0.6, label=f"C{cluster_id}"
            )
        ax.set_title(name)
        ax.legend(markerscale=3, fontsize=8, loc="best")

    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    plt.savefig(os.path.join(out_dir, f"{sc_name}_umap_final.png"), dpi=150)
    plt.close()
    log.info(f"UMAP 2D plot saved — {sc_name}")


def build_cluster_signatures(df_valid, c_bio, c_multi, c_quanti, df_total_len):
    """
    Builds a cluster signature DataFrame with:
      - Binary bio feature frequency (%)
      - Top 3 most frequent categorical imaging values per cluster
      - Quantitative features mean ± std
      - Patient count and weight
    """
    parts = []

    if c_bio:
        parts.append(df_valid.groupby("cluster")[c_bio].mean() * 100)

    for col in c_multi:
        ct = pd.crosstab(df_valid["cluster"], df_valid[col], normalize="index") * 100
        def get_top3(row):
            top = row.sort_values(ascending=False).head(3)
            return " | ".join([f"{n} ({v:.1f}%)" for n, v in top.items() if v > 0])
        parts.append(pd.DataFrame(ct.apply(get_top3, axis=1), columns=[f"top3_{col}"]))

    if c_quanti:
        quanti_df = pd.DataFrame(index=df_valid.groupby("cluster")[c_quanti].mean().index)
        for col in c_quanti:
            m = df_valid.groupby("cluster")[col].mean().round(1)
            s = df_valid.groupby("cluster")[col].std().round(1)
            quanti_df[f"{col}_mean±std"] = m.astype(str) + " ± " + s.astype(str)
        parts.append(quanti_df)

    signatures               = pd.concat(parts, axis=1) if parts else pd.DataFrame()
    counts                   = df_valid["cluster"].value_counts()
    signatures["n_patients"] = counts
    signatures["weight_pct"] = (counts / df_total_len * 100).round(1)
    if c_bio:
        signatures[c_bio]    = signatures[c_bio].round(1)

    return signatures.sort_values("n_patients", ascending=False)

In [6]:
# ============================================================
# 6. DATA LOADING
# ============================================================
df = pd.read_csv(CSV_PATH)
log.info(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

2026-05-20 10:40:37,641 — INFO — Dataset loaded: 56784 rows, 189 columns


In [7]:
# ============================================================
# BERTOPIC STEP 1 : sweep
# ============================================================


for sc_name, sc_def in SCENARIO_CONFIGS.items():
    config      = sc_def["config"]
    active_cols = sc_def["cols"]


    log.info(f"\n{'='*60}\nSCENARIO: {sc_name}\n{'='*60}")
    sc_dir = os.path.join(OUTPUT_DIR, sc_name)
    os.makedirs(sc_dir, exist_ok=True)

    # ── build text ───────────────────────────────────
    df['clinical_text'] = df.apply(
    lambda row, cfg=config, cols=active_cols: build_clinical_text(row, cfg, cols), axis=1
    )
    n_empty = (df['clinical_text'] == "No significant resource utilization was documented.").sum()
    log.info(f"[{sc_name}] Text built — empty: {n_empty} ({100 * n_empty / len(df):.1f}%)")
    log.info(f"[{sc_name}] Example:\n{df['clinical_text'].iloc[0]}")

    # ── Cell 7 : embeddings + UMAP + sweep ───────────────────
    texts      = df['clinical_text'].tolist()
    embeddings = embed_model.encode(texts, batch_size=BATCH_SIZE, show_progress_bar=True)
    np.save(os.path.join(sc_dir, "embeddings.npy"), embeddings)

    umap_embeddings = umap.UMAP(
        n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM,
        min_dist=UMAP_MIN_DIST, metric="cosine", random_state=42
    ).fit_transform(embeddings)
    np.save(os.path.join(sc_dir, "umap_embeddings.npy"), umap_embeddings)

    N    = len(df)
    grid = make_mcs_ms_grid(N)
    log.info(f"[{sc_name}] Grid sweep: {len(grid)} combinations for N={N:,}")
    log.info(f"  mcs range: {int(N*MCS_PCT_VALUES[0]):,} – {int(N*MCS_PCT_VALUES[-1]):,}")

    df_sweep = run_grid_sweep(umap_embeddings, sc_name, sc_dir, N)
    print("\nTop 10 combinations by combined score:")
    print(df_sweep.sort_values("combined_score", ascending=False).head(10).to_string())





2026-05-20 10:40:37,775 — INFO — 
SCENARIO: scenario_2
2026-05-20 10:40:39,883 — INFO — [scenario_2] Text built — empty: 0 (0.0%)
2026-05-20 10:40:39,884 — INFO — [scenario_2] Example:
The patient underwent a CT scan and an X-ray. Blood work was measured. In total, the patient had 2 imaging exam(s) and 1 biological exam(s). The patient was admitted to the hospital.


Batches:   0%|          | 0/222 [00:00<?, ?it/s]

2026-05-20 10:43:44,755 — INFO — [scenario_2] Grid sweep: 40 combinations for N=56,784
2026-05-20 10:43:44,759 — INFO —   mcs range: 1,135 – 3,407
2026-05-20 10:43:44,759 — INFO — [scenario_2] Grid sweep: 40 combinations for N=56,784
2026-05-20 10:43:44,760 — INFO —   mcs range: 1,135 – 3,407
Sweep — scenario_2: 100%|██████████| 40/40 [02:05<00:00,  3.14s/it]
2026-05-20 10:45:51,435 — INFO — [scenario_2] Sweep complete — best: mcs=2839 | ms=567 | clusters=2 | combined=0.5534
2026-05-20 10:45:51,441 — INFO — 
SCENARIO: scenario_3



Top 10 combinations by combined score:
     mcs  mcs_pct_N   ms  ms_pct_mcs  n_clusters  outlier_rate  outlier_pct  silhouette  stability  combined_score
31  2839        5.0  567       19.97           2      0.000000          0.0      0.3089     0.4524          0.5534
39  3407        6.0  681       19.99           2      0.000000          0.0      0.3028     0.4598          0.5495
23  2271        4.0  454       19.99           2      0.000000          0.0      0.3126     0.4326          0.5466
38  3407        6.0  340        9.98           2      0.000000          0.0      0.3067     0.3996          0.5189
29  2839        5.0  141        4.97           2      0.000000          0.0      0.3108     0.3809          0.5133
37  3407        6.0  170        4.99           2      0.015321          1.5      0.3114     0.3852          0.5129
36  3407        6.0   34        1.00           2      0.000000          0.0      0.3170     0.3565          0.5072
28  2839        5.0   28        0.99    

2026-05-20 10:45:57,217 — INFO — [scenario_3] Text built — empty: 0 (0.0%)
2026-05-20 10:45:57,218 — INFO — [scenario_3] Example:
The patient underwent a ct head / brain and a xray emergency survey. Hemoglobin, white blood cell count, differential leukocyte count, blood urea, creatinine, sodium, potassium, platelet count, prothrombin time, aptt, calcium, crp, fibrinogen and blood work were measured. In total, the patient had 2 imaging exam(s) and 1 biological exam(s). The patient was admitted to the hospital.


Batches:   0%|          | 0/222 [00:00<?, ?it/s]

2026-05-20 10:49:00,751 — INFO — [scenario_3] Grid sweep: 40 combinations for N=56,784
2026-05-20 10:49:00,753 — INFO —   mcs range: 1,135 – 3,407
2026-05-20 10:49:00,754 — INFO — [scenario_3] Grid sweep: 40 combinations for N=56,784
2026-05-20 10:49:00,754 — INFO —   mcs range: 1,135 – 3,407
Sweep — scenario_3: 100%|██████████| 40/40 [02:22<00:00,  3.57s/it]
2026-05-20 10:51:24,567 — INFO — [scenario_3] Sweep complete — best: mcs=1135 | ms=113 | clusters=5 | combined=0.3730



Top 10 combinations by combined score:
     mcs  mcs_pct_N   ms  ms_pct_mcs  n_clusters  outlier_rate  outlier_pct  silhouette  stability  combined_score
6   1135        2.0  113        9.96           5      0.592315         59.2      0.6547     0.0924          0.3730
37  3407        6.0  170        4.99           2      0.000634          0.1      0.3739     0.4193          0.3720
7   1135        2.0  227       20.00           5      0.590131         59.0      0.6348     0.1062          0.3671
39  3407        6.0  681       19.99           2      0.005406          0.5      0.3798     0.3999          0.3620
38  3407        6.0  340        9.98           2      0.000634          0.1      0.3728     0.3971          0.3556
36  3407        6.0   34        1.00           2      0.001374          0.1      0.3733     0.3580          0.3284
35  3407        6.0   15        0.44           2      0.063451          6.3      0.4075     0.3345          0.3242
27  2839        5.0   15        0.53    

In [ ]:
# ============================================================
#  FINAL BERTOPIC RUN - step 2
# ============================================================

# ── Manually chosen parameters after sweep inspection ────────────────────
FINAL_PARAMS = {
    "scenario_2": (0.06, 0.97),   # (mcs_pct_N, ms_pct_mcs) — to fill after sweep
    "scenario_3": (0.06, 0.81),
}

for sc_name, sc_def in SCENARIO_CONFIGS.items():
    config      = sc_def["config"]
    active_cols = sc_def["cols"]
    sc_dir      = os.path.join(OUTPUT_DIR, sc_name)

    # Reload UMAP embeddings from sweep
    umap_embeddings = np.load(os.path.join(sc_dir, "umap_embeddings.npy"))
    texts           = df['clinical_text'].tolist()
    N               = len(df)

    # Convert percentages to absolute values
    mcs_pct, ms_pct = FINAL_PARAMS[sc_name]
    BEST_MCS = max(10, int(N * mcs_pct))
    BEST_MS  = max(1, int(BEST_MCS * ms_pct))
    log.info(f"[{sc_name}] Manual params: mcs={BEST_MCS}, ms={BEST_MS}")

    # ── Final BERTopic run ───────────────────────────────────
    topic_model = BERTopic(
        umap_model=umap.UMAP(
            n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM,
            min_dist=UMAP_MIN_DIST, metric="cosine", random_state=42
        ),
        hdbscan_model=hdbscan.HDBSCAN(
            min_cluster_size=BEST_MCS, min_samples=BEST_MS,
            metric="euclidean", gen_min_span_tree=True,
            prediction_data=True
        ),
        vectorizer_model=CountVectorizer(
        ngram_range=(1, 2),
        stop_words=CLINICAL_STOP_WORDS,
        min_df=1,
        max_df=1.0,
        ),
        ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
        embedding_model=embed_model,
        nr_topics="none",
        verbose=True,
        )
    topics, probs    = topic_model.fit_transform(texts)
    df['topic']      = topics
    df['topic_prob'] = [p.max() if hasattr(p, 'max') else p for p in probs]

    topic_model.save(os.path.join(sc_dir, "bertopic_model"))
    df.to_csv(os.path.join(sc_dir, "df_with_topics.csv"), index=False)
    topic_model.get_topic_info().to_csv(os.path.join(sc_dir, "topic_info.csv"), index=False)

    # ── Visualizations ───────────────────────────────────────
    out_dir  = os.path.join(sc_dir, "visualizations")
    os.makedirs(out_dir, exist_ok=True)
    df_valid            = df[df['topic'] != -1].copy()
    df_valid['cluster'] = df_valid['topic']

    c_bio    = [c for c in cols_bio_bin   if c in df_valid.columns]
    c_multi  = [c for c in cols_multi_cat if c in df_valid.columns]
    c_quanti = [c for c in cols_quanti    if c in df_valid.columns]

    plot_wordclouds(df_valid, sc_name, out_dir)
    if c_bio:
        plot_heatmap(df_valid, c_bio, sc_name, out_dir)
        plot_radar(df_valid, c_bio, sc_name, out_dir)

    # Build config dict for UMAP plot title
    n_clusters = len(df_valid['topic'].unique())
    best_config_dict = {
        "n_clusters":   n_clusters,
        "mcs":          BEST_MCS,
        "silhouette":   silhouette_score(
                            umap_embeddings[df['topic'].values >= 0],
                            df['topic'].values[df['topic'].values >= 0],
                            sample_size=min(3000, np.sum(df['topic'].values >= 0))
                        ),
        "stability":    float(np.mean(topic_model.hdbscan_model.cluster_persistence_)),
        "outlier_rate": float(np.mean(df['topic'].values == -1)),
    }
    plot_umap2d(umap_embeddings, df['topic'].values, best_config_dict, sc_name, out_dir)

    signatures = build_cluster_signatures(df_valid, c_bio, c_multi, c_quanti, len(df))
    signatures.to_csv(os.path.join(out_dir, "cluster_signatures.csv"))

    log.info(f"[{sc_name}] Complete — results saved to {sc_dir}")

log.info("All scenarios complete!")

2026-05-20 10:51:24,930 — INFO — [scenario_2] Manual params: mcs=3407, ms=3304
2026-05-20 10:51:24,942 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1775 [00:00<?, ?it/s]

## BERTopic Scenario 2 — Sweep Results (sentence-transformers/embeddinggemma-300m-medical) & Justification for Tabular Approach

BERTopic performs poorly on this dataset compared to the tabular clustering pipeline.
Stability scores are consistently low (max ~0.24 vs 1.0 in tabular s2_balanced), and
silhouette scores never exceed 0.24, indicating heavily overlapping clusters. The best
combined scores are associated with degenerate solutions (2–3 clusters only), which are
clinically unusable. One configuration (mcs=1547, ms=309) produces 0 clusters entirely.
*(results even worse with yikuan8/Clinical-Longformer)*

### Why stability and silhouette trade off in BERTopic

HDBSCAN stability (cluster_persistence_) measures **local density** — a stable cluster
exists across a wide range of density thresholds, meaning its points are tightly packed.
Silhouette, on the other hand, measures both internal cohesion **and inter-cluster
separation** — a good score requires clusters to be dense AND well separated in absolute
distance. In the BERTopic embedding space (text → sentence transformer → UMAP), clusters
can be locally dense but spatially close to one another, which penalizes silhouette while
preserving high stability — or vice versa.

### Why BERTopic underperforms on this dataset

Two structural reasons explain these results:

1. **Text representation is a lossy transformation of tabular data.** Clinical texts here
are short, stereotyped, and built from a restricted vocabulary. Two clinically distinct
patients can produce near-identical sentences (e.g. *"The patient underwent an X-ray.
Blood work was measured. The patient was discharged."*), leaving little signal for the
embedding model to exploit.

2. **BERTopic is designed for long, varied, free-text corpora** (articles, clinical notes,
social media). Programmatically-generated sentences from binary variables are exactly the
type of input that tabular clustering handles better directly — converting tabular data
to text and back through a language model introduces an unnecessary abstraction layer
that loses information rather than adding it.

### Methodological justification for retaining the tabular approach

Beyond metrics, the tabular pipeline offers better **clinical traceability**: the
Hamming + Manhattan distance matrix has a direct clinical interpretation (what variables
differ between two patients), whereas BERTopic embeddings depend on the internal
representations of a language model — a black box harder to justify in an ED triage
context. BERTopic is therefore not retained as the primary clustering approach.